# Python on HPC

You just SSH'd into a supercomputer, launched Jupyter from the command line, and tunneled it to your browser. Let's make sure it was worth it.

---
## Part 0: Prove You're on HPC

In [ ]:
import os
import platform
import time
import numpy as np
%matplotlib inline

print(f"Hostname:  {platform.node()}")
print(f"CPU cores: {os.cpu_count()}")
print(f"Platform:  {platform.machine()}")

with open('/proc/meminfo') as f:
    mem_kb = int(f.readline().split()[1])
    print(f"RAM:       {mem_kb / 1e6:.0f} GB")

gpu_check = os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null').read().strip()
if gpu_check:
    print(f"GPU:       {gpu_check}")
else:
    print("GPU:       Not detected (CPU-only node)")

You should see:
- A hostname like `c###-###.vista.tacc.utexas.edu`
- **72 CPU cores** (ARM-based Grace CPU)
- **223 GB of RAM**
- **NVIDIA GH200 120GB GPU** with ~98 GB of GPU memory

Compare that to your laptop. This is what you're working with today.

---
## Part 1: Matrix Multiplication

Matrix multiplication is everywhere: machine learning, physics simulations, graphics, data analysis. The work scales with the cube of the matrix size -- double the size, 8x the work.

Let's start small and scale up.

In [ ]:
# Small matrix: your laptop could do this
n = 100
A = np.random.randn(n, n)
B = np.random.randn(n, n)

start = time.time()
C = A @ B
elapsed = time.time() - start

print(f"{n}x{n} matrix multiply: {elapsed:.6f} seconds")
print(f"Operations: {2 * n**3:,}")
print("Your laptop handles this fine.")

In [ ]:
# Now scale up
sizes = [100, 500, 1000, 2000, 5000, 10000]
results = []

print(f"{'Size':>8} {'Time (sec)':>12} {'Operations':>18} {'GFLOPS':>10}")
print("-" * 52)

for n in sizes:
    A = np.random.randn(n, n)
    B = np.random.randn(n, n)
    
    start = time.time()
    C = A @ B
    elapsed = time.time() - start
    
    flops = 2 * n**3
    gflops = flops / elapsed / 1e9
    
    results.append((n, elapsed, gflops))
    print(f"{n:>8} {elapsed:>12.4f} {flops:>18,} {gflops:>10.1f}")

In [ ]:
import matplotlib.pyplot as plt

sizes_plot = [r[0] for r in results]
times_plot = [r[1] for r in results]
gflops_plot = [r[2] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(sizes_plot, times_plot, 'o-', color='maroon', linewidth=2, markersize=8)
axes[0].set_xlabel('Matrix Size (n x n)', fontsize=12)
axes[0].set_ylabel('Time (seconds)', fontsize=12)
axes[0].set_title('Time vs Matrix Size', fontsize=14)
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(sizes_plot)), gflops_plot, color='maroon')
axes[1].set_xticks(range(len(sizes_plot)))
axes[1].set_xticklabels([str(s) for s in sizes_plot])
axes[1].set_xlabel('Matrix Size', fontsize=12)
axes[1].set_ylabel('GFLOPS (billions of ops/sec)', fontsize=12)
axes[1].set_title('Compute Throughput', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nThe 10,000x10,000 multiply did {2 * 10000**3 / 1e12:.1f} trillion operations.")

### YOUR TURN

**1. At what size does the multiply take more than 1 second?**

_Your answer:_


**2. Look at the GFLOPS chart. Why does throughput increase with larger matrices?**

_Your answer:_



---
## Part 2: Memory Matters

A 10,000x10,000 matrix of 64-bit floats uses 800 MB. Two of them plus the result = 2.4 GB. Your laptop with 8 GB of RAM can probably handle that.

But what about 20,000x20,000? That's 9.6 GB just for three matrices. Most laptops would struggle. This node has 223 GB of RAM.

In [ ]:
n = 20000
mem_per_matrix = n * n * 8 / 1e9  # 8 bytes per float64
total_mem = mem_per_matrix * 3  # A, B, and result C

print(f"Matrix size: {n:,} x {n:,}")
print(f"Memory per matrix: {mem_per_matrix:.1f} GB")
print(f"Total memory needed (A + B + C): {total_mem:.1f} GB")
print(f"This node has {mem_kb / 1e6:.0f} GB of RAM. No problem.")
print(f"Your laptop probably has 8-16 GB. This would be tight.")
print()

print("Running...")
A = np.random.randn(n, n)
B = np.random.randn(n, n)

start = time.time()
C = A @ B
elapsed = time.time() - start

flops = 2 * n**3
gflops = flops / elapsed / 1e9
print(f"Time: {elapsed:.1f} seconds")
print(f"Operations: {flops / 1e12:.1f} trillion")
print(f"Throughput: {gflops:.0f} GFLOPS")

# Clean up
del A, B, C

### YOUR TURN

**1. How much total RAM does the 20,000x20,000 multiply need? Would your laptop handle it?**

_Your answer:_


**2. What happens if you try to create a matrix that's bigger than your available RAM?**

_Your answer:_



---
## Part 3: Using Multiple Cores

NumPy's matrix multiply already uses multiple cores through BLAS libraries. But what about your own code?

Python's `multiprocessing` lets you split work across all the cores on this node.

In [ ]:
import subprocess

TOTAL = 100_000_000  # 100 million samples

# Python script that runs the Monte Carlo estimation
script = '''
import numpy as np, sys, time
n = int(sys.argv[1])
cores = int(sys.argv[2])

def estimate_pi(n_samples):
    rng = np.random.default_rng()
    x = rng.random(n_samples)
    y = rng.random(n_samples)
    return int(np.sum(x**2 + y**2 <= 1.0))

# Single core
start = time.time()
inside = estimate_pi(n)
t1 = time.time() - start
pi1 = 4.0 * inside / n
print(f"1 core:  pi={pi1:.6f}  time={t1:.2f}s")

# All cores using multiprocessing
from multiprocessing import Pool
chunk = n // cores
start = time.time()
with Pool(cores) as pool:
    results = pool.map(estimate_pi, [chunk] * cores)
t_all = time.time() - start
pi_all = 4.0 * sum(results) / (chunk * cores)
print(f"{cores} cores: pi={pi_all:.6f}  time={t_all:.2f}s")
print(f"")
print(f"Speedup: {t1 / t_all:.1f}x")
print(f"Actual pi: {np.pi:.6f}")
'''

with open('/tmp/mc_pi.py', 'w') as f:
    f.write(script)

n_cores = os.cpu_count()
print(f"Monte Carlo pi estimation: {TOTAL:,} samples")
print(f"Comparing 1 core vs {n_cores} cores")
print()

result = subprocess.run(
    ['python3', '/tmp/mc_pi.py', str(TOTAL), str(n_cores)],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

### YOUR TURN

**What speedup did you get? Why isn't it exactly 72x?**

_Your answer:_



---
## Exit Ticket

Submit these answers on Canvas.

**1. What is the hostname of the compute node you used today?**

_Your answer:_


**2. How many CPU cores and how much RAM does this node have?**

_Your answer:_


**3. How long did the 10,000x10,000 matrix multiply take?**

_Your answer:_


**4. In one sentence: what did you learn today about why researchers use HPC?**

_Your answer:_

